In [2]:
import pandas as pd
import re
from IPython import display

# 文件夹到dE/Ebind解释的映射，用于不同df类型解释不同
def get_colname_map(folder):
    # 默认的解释
    generic_colname_map = {
        "Phase": "Phase",
        "Run": "Run",
        "Method": "Method",
        "BSSE": "BSSE",

        # 反应能相关
        "dE_MImH_MIm_H": "MImH → MIm⁻ + H⁺",
        "dE_1Zn_1MImH_3MeOH_1Zn_1MIm_3MeOH_H": "[Zn·MImH·MeOH₃]²⁺ → [Zn·MIm·MeOH₃]⁺ + H⁺",
        "dE_MImH_H2O_MIm_H3O": "MImH + H₂O → MIm⁻ + H₃O⁺",
        "dE_1Zn_1MImH_3MeOH_H2O_1Zn_1MIm_3MeOH_H3O": "[Zn·MImH·MeOH₃]²⁺ + H₂O → [Zn·MIm·MeOH₃]⁺ + H₃O⁺",
        "dE_H2O_H_H3O": "H₂O + H⁺ → H₃O⁺",
        # 针对 MeOH-Ligand-Exchange
        "dE_1Zn_1MImH_3MeOH_1Zn_1MImH_2MeOH_H2O": "[Zn·MImH·MeOH₃]²⁺ → [Zn·MImH·MeOH₂·H₂O]²⁺ + MeOH",
        "dE_1Zn_1MIm_3MeOH_1Zn_1MIm_2MeOH_H2O": "[Zn·MIm·MeOH₃]⁺ → [Zn·MIm·MeOH₂·H₂O]⁺ + MeOH",
        "dE_1Zn_1MImH_2MeOH_H2O_1Zn_1MIm_2MeOH_H2O_H": "[Zn·MImH·MeOH₂·H₂O]²⁺ → [Zn·MIm·MeOH₂·H₂O]⁺ + H⁺",
        "dE_1Zn_1MImH_2MeOH_H2O_H2O_1Zn_1MIm_2MeOH_2H2O_H3O": "[Zn·MImH·MeOH₂·H₂O]²⁺ + H₂O → [Zn·MIm·MeOH₂·2H₂O]⁺ + H₃O⁺"
    }

    # MIMH: Zn配体逐步置换（从0 MImH到4 MImH）
    MIMH_step_de = {
        "dE_0_1": "[Zn·MeOH₄]²⁺ + MImH → [Zn·MImH·MeOH₃]²⁺ + MeOH",
        "dE_1_2": "[Zn·MImH·MeOH₃]²⁺ + MImH → [Zn·(MImH)₂·MeOH₂]²⁺ + MeOH",
        "dE_2_3": "[Zn·(MImH)₂·MeOH₂]²⁺ + MImH → [Zn·(MImH)₃·MeOH]²⁺ + MeOH",
        "dE_3_4": "[Zn·(MImH)₃·MeOH]²⁺ + MImH → [Zn·(MImH)₄]²⁺ + MeOH",

        "Ebind_1Zn_0MImH_4MeOH": "[Zn·MeOH₄]²⁺ Ebind",
        "Ebind_1Zn_1MImH_3MeOH": "[Zn·MImH·MeOH₃]²⁺ Ebind",
        "Ebind_1Zn_2MImH_2MeOH": "[Zn·(MImH)₂·MeOH₂]²⁺ Ebind",
        "Ebind_1Zn_3MImH_1MeOH": "[Zn·(MImH)₃·MeOH]²⁺ Ebind",
        "Ebind_1Zn_4MImH_0MeOH": "[Zn·(MImH)₄]²⁺ Ebind",
    }

    # MIM: Zn配体逐步置换（从0 MIm到4 MIm，单正离子）
    MIM_step_de = {
        "dE_0_1": "[Zn·MeOH₄]⁺ + MIm → [Zn·MIm·MeOH₃]⁺ + MeOH",
        "dE_1_2": "[Zn·MIm·MeOH₃]⁺ + MIm → [Zn·(MIm)₂·MeOH₂]⁺ + MeOH",
        "dE_2_3": "[Zn·(MIm)₂·MeOH₂]⁺ + MIm → [Zn·(MIm)₃·MeOH]⁺ + MeOH",
        "dE_3_4": "[Zn·(MIm)₃·MeOH]⁺ + MIm → [Zn·(MIm)₄]⁺ + MeOH",

        "Ebind_1Zn_0MImH_4MeOH": "[Zn·MeOH₄]⁺ Ebind",
        "Ebind_1Zn_1MImH_3MeOH": "[Zn·MIm·MeOH₃]⁺ Ebind",
        "Ebind_1Zn_2MImH_2MeOH": "[Zn·(MIm)₂·MeOH₂]⁺ Ebind",
        "Ebind_1Zn_3MImH_1MeOH": "[Zn·(MIm)₃·MeOH]⁺ Ebind",
        "Ebind_1Zn_4MImH_0MeOH": "[Zn·(MIm)₄]⁺ Ebind",
    }

    # deprotonate: 只用默认映射，不管step dE/Ebind
    if "MIMH_PBE_STRUC" in folder and "lig_exchange" in folder:
        return {**generic_colname_map, **MIMH_step_de}
    elif "MIM_PBE_STRUC" in folder and "lig_exchange" in folder:
        return {**generic_colname_map, **MIM_step_de}
    else:
        return generic_colname_map

folder_ls = ['lig_exchange/MIMH_PBE_STRUC/', 'lig_exchange/MIM_PBE_STRUC/', 'deprotonate/MIMH_PBE_STRUC/']
for folder in folder_ls:
    df = pd.read_csv(f'{folder}/dE.csv')
    # Drop columns whose names start with "E_"
    df = df[[col for col in df.columns if not col.startswith("E_")]]
    # Drop rows where "Phase" is "SP_init" and "Method" contains DFTB, xTB, or MM
    if "Phase" in df.columns and "Method" in df.columns:
        mask = ~(
            (df["Phase"] == "SP_init") &
            (df["Method"].str.contains('DFTB|xTB|MM', flags=re.IGNORECASE, na=False))
        )
        df = df[mask]
    # Drop, for each Method, the rows with BSSE='no' if BSSE='yes' exists for the same Method
    if 'BSSE' in df.columns and 'Method' in df.columns:
        methods_with_yes = set(df.loc[df["BSSE"]=="yes", "Method"])
        mask_keep = ~(
            (df["Method"].isin(methods_with_yes)) & (df["BSSE"] == "no")
        )
        df = df[mask_keep]
    # Drop rows where Run is in ['run18', 'run19', 'run20', 'run21', 'run14']
    if 'Run' in df.columns:
        df = df[~df['Run'].isin(['run18', 'run19', 'run20', 'run21','run14'])]

    # ----------- Custom column re-order: Ebind after all dE columns ----------- #
    # Find non-dE columns, dE columns, Ebind columns
    all_cols = list(df.columns)
    dE_cols = [col for col in all_cols if col.startswith("dE_")]
    Ebind_cols = [col for col in all_cols if col.startswith("Ebind_")]
    # All other columns (not dE or Ebind)
    other_cols = [col for col in all_cols if not (col.startswith("dE_") or col.startswith("Ebind_"))]
    # The new order: others + dE_cols + Ebind_cols
    new_order = other_cols + dE_cols + Ebind_cols
    # Only use columns that actually exist in df (just in case)
    new_order_existing = [col for col in new_order if col in df.columns]
    df = df[new_order_existing]
    # ------------------------------------------------------------------------- #

    # 保留一位小数
    for col in df.columns:
        # 尝试只对数值型进行保留一位小数
        if pd.api.types.is_float_dtype(df[col]) or pd.api.types.is_integer_dtype(df[col]):
            df[col] = df[col].round(1)

    # 按照BSSE后第一个列降序排列
    if 'BSSE' in df.columns:
        bsse_idx = list(df.columns).index('BSSE')
        if bsse_idx+1 < len(df.columns):
            first_after_bsse = df.columns[bsse_idx+1]
            df = df.sort_values(first_after_bsse, ascending=False, ignore_index=True)

    # 替换列名为化学式/化学方程式
    colname_map = get_colname_map(folder)
    rename_dict = {}
    for col in df.columns:
        if col in colname_map:
            rename_dict[col] = colname_map[col]
        # dE/Ebind/自定义未枚举的dE列，可做自动变换为化学式,否则保留原名
    df = df.rename(columns=rename_dict)

    display.display(df)

    # 将处理后的每个df分别存储到各自目录下的selected_dE.csv
    df.to_csv(f'{folder}/selected_dE.csv', index=False, encoding="utf-8-sig")

,Phase,Run,Method,BSSE,[Zn·MeOH₄]²⁺ + MImH → [Zn·MImH·MeOH₃]²⁺ + MeOH,[Zn·MImH·MeOH₃]²⁺ + MImH → [Zn·(MImH)₂·MeOH₂]²⁺ + MeOH,[Zn·(MImH)₂·MeOH₂]²⁺ + MImH → [Zn·(MImH)₃·MeOH]²⁺ + MeOH,[Zn·(MImH)₃·MeOH]²⁺ + MImH → [Zn·(MImH)₄]²⁺ + MeOH,[Zn·MeOH₄]²⁺ Ebind,[Zn·MImH·MeOH₃]²⁺ Ebind,[Zn·(MImH)₂·MeOH₂]²⁺ Ebind,[Zn·(MImH)₃·MeOH]²⁺ Ebind,[Zn·(MImH)₄]²⁺ Ebind
0,SP_opt,run0,MM,no,-20.0,-19.7,-19.7,-19.5,-256.6,-276.7,-296.4,-316.1,-335.6
1,SP_opt,run5,MM OPC,no,-20.6,-20.0,-20.1,-19.7,-279.6,-300.1,-320.1,-340.2,-359.9
2,SP_init,run24,DLPNO-CCSD(T) RIJCOSX def2-TZVPPD def2-TZVPPD/...,yes,-35.0,-30.7,-28.6,-25.3,-305.0,-340.0,-370.7,-399.2,-424.5
3,SP_init,run22,RI-MP2 RIJCOSX def2-TZVPPD def2-TZVPPD/C AutoAux,yes,-35.4,-31.4,-29.3,-26.4,-307.9,-343.3,-374.7,-404.1,-430.5
4,SP_init,run13,M06L-D3 def2-TZVPPD DEFGRID3,yes,-36.3,-32.6,-28.5,-25.0,-312.0,-348.3,-380.8,-409.3,-434.3
5,SP_init,run23,B3LYP def2-TZVPPD D3BJ AutoAux,yes,-36.5,-31.7,-29.0,-25.7,-327.3,-363.8,-395.5,-424.5,-450.2
6,SP_opt,run27,g-xTB,no,-36.6,-33.8,-30.7,-27.7,-321.5,-358.1,-391.9,-422.6,-450.3
7,SP_opt,run8,mDFTB3D3/3ob_prime,no,-37.0,-32.7,-29.6,-27.1,-340.3,-377.4,-410.1,-439.6,-466.7
8,SP_init,run16,PBE-D3(BJ) def2-TZVPPD,yes,-37.7,-32.5,-29.3,-25.6,-326.2,-363.9,-396.3,-425.6,-451.2
9,SP_opt,run2,DFTB3-D3H5,no,-37.8,-33.8,-30.1,-27.1,-334.1,-371.9,-405.7,-435.8,-462.9


,Phase,Run,Method,BSSE,[Zn·MeOH₄]⁺ + MIm → [Zn·MIm·MeOH₃]⁺ + MeOH,[Zn·MIm·MeOH₃]⁺ + MIm → [Zn·(MIm)₂·MeOH₂]⁺ + MeOH,[Zn·(MIm)₂·MeOH₂]⁺ + MIm → [Zn·(MIm)₃·MeOH]⁺ + MeOH,[Zn·(MIm)₃·MeOH]⁺ + MIm → [Zn·(MIm)₄]⁺ + MeOH,Ebind_1Zn_0MIm_4MeOH,Ebind_1Zn_1MIm_3MeOH,Ebind_1Zn_2MIm_2MeOH,Ebind_1Zn_3MIm_1MeOH,Ebind_1Zn_4MIm_0MeOH
0,SP_init,run22,RI-MP2 RIJCOSX def2-TZVPPD def2-TZVPPD/C AutoAux,yes,-197.8,-125.7,-50.3,14.5,-307.9,-505.7,-631.4,-681.7,-667.3
1,SP_init,run24,DLPNO-CCSD(T) RIJCOSX def2-TZVPPD def2-TZVPPD/...,yes,-198.0,-125.4,-50.2,14.9,-305.0,-502.9,-628.3,-678.5,-663.6
2,SP_init,run23,B3LYP def2-TZVPPD D3BJ AutoAux,yes,-200.1,-125.8,-49.2,15.6,-327.3,-527.4,-653.2,-702.5,-686.8
3,SP_init,run13,M06L-D3 def2-TZVPPD DEFGRID3,yes,-202.7,-128.7,-48.7,16.7,-312.0,-514.6,-643.3,-692.1,-675.4
4,SP_init,run16,PBE-D3(BJ) def2-TZVPPD,yes,-202.9,-126.5,-48.3,16.6,-326.2,-529.1,-655.7,-704.0,-687.3
5,SP_opt,run27,g-xTB,no,-204.6,-127.2,-54.5,13.4,-321.5,-526.1,-653.3,-707.7,-694.3
6,SP_opt,run2,DFTB3-D3H5,no,-214.7,-136.3,-54.8,14.2,-334.1,-548.8,-685.2,-740.0,-725.8
7,SP_opt,run1,DFTB3-D3,no,-214.7,-136.5,-55.3,13.6,-330.3,-545.0,-681.5,-736.8,-723.2
8,SP_opt,run17,DFTB3 DampingH,no,-215.2,-135.4,-54.2,15.6,-319.8,-535.0,-670.5,-724.7,-709.1
9,SP_opt,run5,MM OPC,no,-216.7,-110.6,-59.0,6.8,-279.6,-496.2,-606.8,-665.9,-659.1


,Phase,Run,Method,BSSE,MImH → MIm⁻ + H⁺,[Zn·MImH·MeOH₃]²⁺ → [Zn·MIm·MeOH₃]⁺ + H⁺,MImH + H₂O → MIm⁻ + H₃O⁺,[Zn·MImH·MeOH₃]²⁺ + H₂O → [Zn·MIm·MeOH₃]⁺ + H₃O⁺,H₂O + H⁺ → H₃O⁺
0,SP_opt,run9,mDFTB3D3/3ob,no,393.3,205.9,216.9,29.4,-176.4
1,SP_opt,run8,mDFTB3D3/3ob_prime,no,384.9,202.7,205.5,23.3,-179.4
2,SP_opt,run15,mDFTB3D3/3ob_prime O_unmodified,no,384.9,202.2,207.1,24.4,-177.8
3,SP_opt,run6,DFTB3-D3 DampingH,no,377.7,200.8,204.9,28.0,-172.8
4,SP_opt,run2,DFTB3-D3H5,no,377.5,200.6,201.1,24.1,-176.5
5,SP_opt,run17,DFTB3 DampingH,no,377.3,200.6,204.6,27.8,-172.7
6,SP_opt,run1,DFTB3-D3,no,377.1,201.2,199.9,24.0,-177.2
7,SP_init,run13,M06L-D3 def2-TZVPPD DEFGRID3,yes,363.3,197.0,188.8,22.4,-174.5
8,SP_init,run23,B3LYP def2-TZVPPD D3BJ AutoAux,yes,361.0,197.4,189.8,26.2,-171.2
9,SP_opt,run27,g-xTB,no,360.1,192.0,190.7,22.7,-169.4
